In [1]:
from pathlib import Path
from collections import Counter
import re

import pandas as pd

DATA_DIR = Path(r"P:\HACKATHONS\SIH26PS67\DATA\REF")
COASTLINE_DIR = DATA_DIR / "coastline"
GSHHG_DIR = COASTLINE_DIR / "gshhg"   # notice the G at the end

GSHHS_DIR = GSHHG_DIR / "GSHHS_shp"   # notice the S at the end
WDBII_DIR = GSHHG_DIR / "WDBII_shp"

In [2]:
# to veryify if the directories exist

paths = {
    "GSHHG": GSHHG_DIR,
    "GSHHS": GSHHS_DIR,
    "WDBII": WDBII_DIR,
}

for name, path in paths.items():
    print(f"{name:6} | {'EXISTS' if path.exists() else 'MISSING'} | {path}")

GSHHG  | EXISTS | P:\HACKATHONS\SIH26PS67\DATA\REF\coastline\gshhg
GSHHS  | EXISTS | P:\HACKATHONS\SIH26PS67\DATA\REF\coastline\gshhg\GSHHS_shp
WDBII  | EXISTS | P:\HACKATHONS\SIH26PS67\DATA\REF\coastline\gshhg\WDBII_shp


In [3]:
RESOLUTIONS = ["c", "l", "i", "h", "f"]

In [4]:
# to verify if resolutions exist

for resolution in RESOLUTIONS:
    path = GSHHS_DIR / resolution
    print(f"{path}: {'EXISTS' if path.exists() else 'MISSING'}")

P:\HACKATHONS\SIH26PS67\DATA\REF\coastline\gshhg\GSHHS_shp\c: EXISTS
P:\HACKATHONS\SIH26PS67\DATA\REF\coastline\gshhg\GSHHS_shp\l: EXISTS
P:\HACKATHONS\SIH26PS67\DATA\REF\coastline\gshhg\GSHHS_shp\i: EXISTS
P:\HACKATHONS\SIH26PS67\DATA\REF\coastline\gshhg\GSHHS_shp\h: EXISTS
P:\HACKATHONS\SIH26PS67\DATA\REF\coastline\gshhg\GSHHS_shp\f: EXISTS


In [5]:
all_files = sorted(
    path for path in GSHHS_DIR.rglob("*")
    if path.is_file()
)

print(f"Total files: {len(all_files):,}")

Total files: 117


In [6]:
extension_counts = Counter(
    path.suffix.lower()
    for path in all_files
)

extension_df = (
    pd.DataFrame(
        extension_counts.items(),
        columns=["extension", "file_count"]
    )
    .sort_values("extension")
    .reset_index(drop=True)
)

extension_df

,extension,file_count
0,.dbf,29
1,.prj,29
2,.shp,29
3,.shx,29
4,.txt,1


In [7]:
gshhs_files = sorted(
    path for path in GSHHS_DIR.rglob("*")
    if path.is_file()
)

wdbii_files = sorted(
    path for path in WDBII_DIR.rglob("*")
    if path.is_file()
)

print(f"GSHHS files: {len(gshhs_files):,}")
print(f"WDBII files: {len(wdbii_files):,}")

GSHHS files: 117
WDBII files: 281


In [8]:
resolution_counts = {}

for resolution in RESOLUTIONS:
    resolution_counts[resolution] = sum(
        1
        for path in gshhs_files
        if path.parent.name == resolution
    )

resolution_df = pd.DataFrame(
    {
        "resolution": RESOLUTIONS,
        "file_count": [
            resolution_counts[r]
            for r in RESOLUTIONS
        ],
    }
)

resolution_df

,resolution,file_count
0,c,20
1,l,24
2,i,24
3,h,24
4,f,24


In [9]:
shp_files = sorted(
    path for path in gshhs_files
    if path.suffix.lower() == ".shp"
)

print(f"Shapefiles: {len(shp_files):,}")

Shapefiles: 29


In [10]:
pattern = re.compile(
    r"GSHHS_([clihf])_L([1-6])\.shp$",
    re.IGNORECASE
)

records = []

for path in shp_files:
    match = pattern.search(path.name)

    if match:
        resolution = match.group(1).lower()
        level = int(match.group(2))

        records.append({
            "resolution": resolution,
            "level": level,
            "filename": path.name,
            "path": path,
        })

dataset_inventory = pd.DataFrame(records)

dataset_matrix = (
    dataset_inventory
    .pivot(
        index="resolution",
        columns="level",
        values="filename"
    )
    .reindex(RESOLUTIONS)
)

dataset_matrix

level,1,2,3,4,5,6
resolution,,,,,,
c,GSHHS_c_L1.shp,GSHHS_c_L2.shp,GSHHS_c_L3.shp,NaN,GSHHS_c_L5.shp,GSHHS_c_L6.shp
l,GSHHS_l_L1.shp,GSHHS_l_L2.shp,GSHHS_l_L3.shp,GSHHS_l_L4.shp,GSHHS_l_L5.shp,GSHHS_l_L6.shp
i,GSHHS_i_L1.shp,GSHHS_i_L2.shp,GSHHS_i_L3.shp,GSHHS_i_L4.shp,GSHHS_i_L5.shp,GSHHS_i_L6.shp
h,GSHHS_h_L1.shp,GSHHS_h_L2.shp,GSHHS_h_L3.shp,GSHHS_h_L4.shp,GSHHS_h_L5.shp,GSHHS_h_L6.shp
f,GSHHS_f_L1.shp,GSHHS_f_L2.shp,GSHHS_f_L3.shp,GSHHS_f_L4.shp,GSHHS_f_L5.shp,GSHHS_f_L6.shp


In [11]:
expected = {
    (resolution, level)
    for resolution in RESOLUTIONS
    for level in range(1, 7)
}

actual = set(
    zip(
        dataset_inventory["resolution"],
        dataset_inventory["level"]
    )
)

missing = sorted(expected - actual)
extra = sorted(actual - expected)

print("Expected datasets :", len(expected))
print("Found datasets    :", len(actual))
print("Missing datasets  :", missing)
print("Unexpected datasets:", extra)

Expected datasets : 30
Found datasets    : 29
Missing datasets  : [('c', 4)]
Unexpected datasets: []
